# Module 1.4: Industrial Data Cleaning & Telemetry Spike Removal
### 🇵🇱 Czyszczenie Danych Przemysłowych i Usuwanie Szpilek Zakłócających

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m01_l04_data_cleaning_spike_removal.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Implement Hampel filtering (rolling median + MAD), forward-fill state imputation, and cubic spline interpolation on corrupted plant telemetry.  
> 🇵🇱 **Streszczenie:** Zaimplementuj filtr Hampela (krocząca mediana + MAD), uzupełnianie w przód (LOCF) i interpolację sklejaną na uszkodzonej telemetrii.

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Industrial Telemetry Glitches**: Sensor cables running near high-voltage VFDs and welding robots pick up transient electromagnetic spikes (thousands of volts for microsecond bursts).
2. **The Hampel Filter**: A robust outlier rejection filter based on rolling Median Absolute Deviation (MAD):
   $$\text{MAD}_t = 1.4826 \cdot \text{median}\left(|x_{t-k \dots t+k} - \tilde{x}_t|\right)$$
   If $|x_t - \tilde{x}_t| > 3 \cdot \text{MAD}_t$, replace $x_t$ with the rolling median $\tilde{x}_t$.
3. **Phase-Preserving Denoising**: Unlike moving average filters which shift waveform peaks and smear spikes into broader hills, the Hampel filter isolates only anomalous samples, preserving sharp physical transients.


In [ ]:
# 1. Simulate Clean Telemetry with Welding Arc Spikes and Packet Drops
np.random.seed(42)
n_pts = 1000
t = np.linspace(0, 10, n_pts)

# Clean thermal signal with diurnal cycle
clean_signal = 65.0 + 12.0 * np.sin(2 * np.pi * 0.2 * t)

# Inject electromagnetic welding spikes (15-40°C transient surges)
corrupted_signal = clean_signal.copy()
spike_indices = np.random.choice(n_pts, size=15, replace=False)
corrupted_signal[spike_indices] += np.random.choice([-1, 1], size=15) * np.random.uniform(25.0, 50.0, size=15)

# Inject sensor packet dropouts (NaNs)
dropout_indices = np.random.choice(n_pts, size=20, replace=False)
corrupted_signal[dropout_indices] = np.nan

# Forward-fill dropouts with limit=3
cleaned_fill = pd.Series(corrupted_signal).ffill(limit=3).bfill().to_numpy()
print(f"✓ Created signal with {len(spike_indices)} spikes and {len(dropout_indices)} dropouts.")


In [ ]:
# 2. Implement Hampel Filter vs Moving Average Filter
def hampel_filter(data, window_size=7, n_sigmas=3.0):
    k = 1.4826  # Normal distribution scale factor
    series = pd.Series(data)
    rolling_median = series.rolling(window=window_size, center=True).median()
    rolling_mad = k * (series - rolling_median).abs().rolling(window=window_size, center=True).median()
    
    threshold = n_sigmas * rolling_mad
    diff = (series - rolling_median).abs()
    outlier_mask = diff > threshold
    
    filtered_series = series.copy()
    filtered_series[outlier_mask] = rolling_median[outlier_mask]
    return filtered_series.to_numpy(), outlier_mask.to_numpy()

# Naive Moving Average
naive_ma = pd.Series(cleaned_fill).rolling(window=11, center=True).mean().bfill().ffill().to_numpy()

# Hampel Cleaned
hampel_cleaned, detected_spikes = hampel_filter(cleaned_fill, window_size=9, n_sigmas=3.0)

plt.figure(figsize=(14, 5))
plt.plot(t, cleaned_fill, color='#ef4444', alpha=0.45, label='Raw Telemetry with Arc Spikes')
plt.plot(t, naive_ma, color='#f59e0b', linestyle='--', linewidth=1.8, label='Naive Moving Average (Smeared Error)')
plt.plot(t, hampel_cleaned, color='#10b981', linewidth=2.0, label='Hampel Filter (Preserves True Dynamic)')
plt.plot(t, clean_signal, color='#0284c7', linestyle=':', linewidth=1.5, label='Ground Truth Physical Process')

plt.title("Industrial Signal Conditioning: Hampel Filter vs Naive Moving Average", fontsize=13, fontweight='bold')
plt.xlabel("Time [seconds]")
plt.ylabel("Temperature [°C]")
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Implement Hampel outlier filter using rolling median and Median Absolute Deviation (MAD) *(🇵🇱 Zaimplementuj filtr Hampela z kroczącą medianą i odchyleniem medianowym MAD)*
- [ ] **Task 2:** Handle intermittent wireless packet dropouts using Last Observation Carried Forward (LOCF) *(🇵🇱 Obsłuż utratę pakietów w sieci bezprzewodowej metodą przenoszenia ostatniej wartości (LOCF))*
- [ ] **Task 3:** Interpolate continuous physical pressure drops using cubic splines without edge ripple *(🇵🇱 Zinterpoluj ubytki ciągłego sygnału ciśnienia funkcjami sklejanymi 3. stopnia)*


In [ ]:
# Exercise Challenge:
# Compute RMSE against ground truth for both filters
rmse_ma = np.sqrt(np.mean((clean_signal - naive_ma)**2))
rmse_hampel = np.sqrt(np.mean((clean_signal - hampel_cleaned)**2))

print(f"Moving Average RMSE : {rmse_ma:.3f}°C")
print(f"Hampel Filter RMSE  : {rmse_hampel:.3f}°C")
assert rmse_hampel < rmse_ma * 0.5, "Hampel filter should deliver at least 2x lower RMSE than moving average."
print("✓ Spike rejection verification passed!")
